In [8]:
pip install torch datasets

In [ ]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_VOCAB_SIZE = 20000
MAX_LEN = 100
BATCH_SIZE = 128
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_CLASSES = 4
NUM_EPOCHS = 5
LEARNING_RATE = 0.001


dataset = load_dataset("ag_news", revision="main", token=True)

train_data = dataset["train"]
test_data = dataset["test"]

label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}


def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text

def tokenize(text):
    return clean_text(text).split()

# Build vocabulary from training data
counter = Counter()
for item in train_data:
    counter.update(tokenize(item["text"]))

most_common_tokens = counter.most_common(MAX_VOCAB_SIZE - 2)
vocab = {"<PAD>": 0, "<UNK>": 1}

for idx, (token, _) in enumerate(most_common_tokens, start=2):
    vocab[token] = idx

def numericalize(text):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens]
    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]
    return ids


class AGNewsTorchDataset(torch.utils.data.Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        text = self.dataset[idx]["text"]
        label = self.dataset[idx]["label"]
        input_ids = torch.tensor(numericalize(text), dtype=torch.long)
        label = torch.tensor(label, dtype=torch.long)
        return input_ids, label

train_dataset = AGNewsTorchDataset(train_data)
test_dataset = AGNewsTorchDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(SimpleRNNClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)              # (batch, seq_len, embedding_dim)
        output, hidden = self.rnn(x)       # hidden: (1, batch, hidden_dim)
        last_hidden = hidden[-1]           # (batch, hidden_dim)
        logits = self.fc(last_hidden)      # (batch, num_classes)
        return logits

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_classes=NUM_CLASSES
).to(DEVICE)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)


def train_model(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate_model(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_model(model, train_loader, optimizer, criterion, DEVICE)
    test_loss, test_acc = evaluate_model(model, test_loader, criterion, DEVICE)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test Loss:  {test_loss:.4f} | Test Acc:  {test_acc:.4f}")
    print("-" * 50)


def predict_news(text, model, device):
    model.eval()
    input_ids = torch.tensor([numericalize(text)], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids)
        pred = torch.argmax(outputs, dim=1).item()

    return label_names[pred]

sample_text = "Apple releases new AI-powered devices in the global market."
prediction = predict_news(sample_text, model, DEVICE)
print("Predicted Category:", prediction)